# 05 — Grey Wolf Optimizer (GWO) Feature Selection

**Goal:** use GWO to search for the feature subset that maximizes classification
performance while minimizing the number of features used. GWO mimics the leadership hierarchy and hunting behavior of grey wolves: the best three solutions (alpha, beta, delta) guide the rest of the pack toward promising regions of the search space.

This notebook follows the exact same pipeline as every other optimizer notebook in
this project (GA / PSO / GWO / WOA) — only the optimizer differs — so results are
directly comparable in `07_Comparison.ipynb`.

**Pipeline:**

```
Initialize wolf pack (candidate feature masks)
        |
        v
  Evaluate fitness (via utils.fitness.fitness)
        |
        v
  Rank wolves -> alpha, beta, delta (3 best solutions)
        |
        v
  Update remaining wolves' positions toward alpha/beta/delta
        |
        v
  Repeat for `epoch` iterations
        |
        v
  Return best feature subset (alpha wolf)
```


In [ ]:
# !pip install -q mealpy

import sys
sys.path.append("..")

import time
import numpy as np
import pandas as pd

from mealpy.swarm_based.GWO import OriginalGWO

from utils.preprocessing import load_processed_data
from utils.metrics import evaluate_subset
from utils.fitness import make_fitness_fn, binarize

## 1. Load processed data (from notebook 01)

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_processed_data("../datasets")
n_features = X_train.shape[1]
print("Number of features:", n_features)

## 2. Define the fitness function

`utils.fitness.fitness` is shared across ALL optimizer notebooks. It combines
classification error with a small penalty on the number of selected features.

In [ ]:
fitness_fn = make_fitness_fn(
    X_train, X_test, y_train, y_test,
    classifier="svm",   # keep consistent with the baseline notebook
    alpha=0.99,
)

# mealpy expects fit_func(solution) -> float (to minimize)
def obj_func(solution):
    return fitness_fn(solution)

## 3. Configure and run GWO

Solutions are continuous vectors in [0, 1]^n_features; each dimension is
binarized (`>0.5` -> feature selected) inside the fitness function.

In [ ]:
EPOCH = 50        # number of iterations/generations
POP_SIZE = 30      # population/swarm size

problem_dict = {
    "fit_func": obj_func,
    "lb": [0.0, ] * n_features,
    "ub": [1.0, ] * n_features,
    "minmax": "min",
    "log_to": None,
}

model = OriginalGWO(epoch=EPOCH, pop_size=POP_SIZE)

start_time = time.time()
best_agent = model.solve(problem_dict)
runtime = time.time() - start_time

best_position = best_agent.solution
best_fitness = best_agent.target.fitness

print("Best fitness:", best_fitness)
print("Runtime (s):", runtime)

## 4. Extract convergence curve

Saved for the comparison plots in `07_Comparison.ipynb`.

In [ ]:
convergence_curve = model.history.list_global_best_fit
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(convergence_curve, linewidth=2)
plt.xlabel("Iteration")
plt.ylabel("Best Fitness")
plt.title("GWO Convergence")
plt.grid(alpha=0.3)
plt.show()

## 5. Evaluate the best feature subset

Using the exact same `evaluate_subset()` function as the baseline notebook.

In [ ]:
best_mask = binarize(best_position)
selected_features = [f for f, m in zip(feature_names, best_mask) if m == 1]

print(f"GWO selected {best_mask.sum()} / {n_features} features:")
print(selected_features)

results = evaluate_subset(best_mask, X_train, X_test, y_train, y_test, classifier="svm")
for k in ["accuracy", "precision", "recall", "f1", "roc_auc", "n_features"]:
    print(f"  {k}: {results[k]}")

## 6. Save results

Saved to `../results/gwo_results.csv` and
`../results/gwo_convergence.npy` for `07_Comparison.ipynb`.

In [ ]:
import os
os.makedirs("../results", exist_ok=True)

results_df = pd.DataFrame([{
    "Algorithm": "GWO",
    "Accuracy": results["accuracy"],
    "Precision": results["precision"],
    "Recall": results["recall"],
    "F1": results["f1"],
    "ROC_AUC": results["roc_auc"],
    "Features": results["n_features"],
    "Runtime": runtime,
}])

results_df.to_csv("../results/gwo_results.csv", index=False)
np.save("../results/gwo_convergence.npy", np.array(convergence_curve))
np.save("../results/gwo_mask.npy", best_mask)

results_df

## Next step

Run the remaining optimizer notebooks (any not yet run), then open
`07_Comparison.ipynb` to compare GWO against the baseline and the other
metaheuristics side by side.